# **Data Cleaning**

## Objectives

* Act on the five issues notebook 01 identified.
* Decide how the disguised missing values are handled, based on evidence rather
  than on habit.
* Save a cleaned dataset that the exploration notebook can load directly.

## Inputs

* `inputs/datasets/raw/BankChurners.csv`

## Outputs

* `outputs/datasets/cleaned/bank_churners_cleaned.csv`

## Approach

The raw file is never modified. Every step below is applied to a copy, so the
whole notebook can be re-run from the original source at any time.

---

# Change working directory

In [1]:
import os
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main\\jupyter_notebooks'

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [3]:
current_dir = os.getcwd()
current_dir

'G:\\Projects\\Credit_Card_Churn-Analysis-main\\Credit_Card_Churn-Analysis-main'

---

# Section 1: Load the raw data

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

raw = pd.read_csv('inputs/datasets/raw/BankChurners.csv')
df = raw.copy()

print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]} columns")

Loaded 10,127 rows and 23 columns


---

# Section 2: Drop columns that cannot be used

Four columns go, for three different reasons. Each was verified in notebook 01
rather than dropped on reputation.

| Column | Reason |
|---|---|
| `CLIENTNUM` | Unique per row. An identifier, not an attribute. |
| `Avg_Open_To_Buy` | Exactly `Credit_Limit` - `Total_Revolving_Bal`. |
| `Naive_Bayes_Classifier_..._1` | Reproduces the target. Leakage. |
| `Naive_Bayes_Classifier_..._2` | As above. |

The leakage columns are the important ones. Leaving them in would not produce an
error - it would produce a confident, completely circular result.

In [5]:
leakage_cols = [col for col in df.columns if col.startswith('Naive_Bayes')]
to_drop = ['CLIENTNUM', 'Avg_Open_To_Buy'] + leakage_cols

df = df.drop(columns=to_drop)

print("Dropped:")
for col in to_drop:
    print(f"  - {col}")
print(f"\nRemaining: {df.shape[1]} columns")

Dropped:
  - CLIENTNUM
  - Avg_Open_To_Buy
  - Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1
  - Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2

Remaining: 19 columns


---

# Section 3: Encode the target

`Attrition_Flag` is text. A 0/1 column is added so that churn rates can be
computed with a mean, which is what every comparison in notebook 03 relies on.

The original text column is kept as well, because it makes plot legends readable
without a lookup.

In [6]:
df['Churn'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)

check = df.groupby('Attrition_Flag')['Churn'].agg(['mean', 'size'])
check

,mean,size
Attrition_Flag,,
Attrited Customer,1.0,1627
Existing Customer,0.0,8500


In [7]:
print(f"Churn rate: {df['Churn'].mean():.4f}")

Churn rate: 0.1607


The mapping is right: every attrited customer scores 1, every existing customer
scores 0, and the overall rate matches the 16.07% found in notebook 01.

---

# Section 4: The `Unknown` values

Notebook 01 found that `Education_Level`, `Income_Category` and
`Marital_Status` use `Unknown` in place of a real value. There are three
possible responses, and choosing between them needs evidence.

1. **Drop the rows.** 30.1% of the file has at least one Unknown. Unacceptable.
2. **Impute** - fill with the most common value, or predict it. This invents
   data. Every imputed value would then appear in the demographic charts as
   though it had been observed.
3. **Keep as explicitly missing** and let each comparison use the customers who
   actually have that attribute recorded.

Option 3 is only safe if the missingness is unrelated to churn. If customers who
withhold their income were also more likely to leave, then excluding them would
quietly bias every demographic result. That is testable.

In [8]:
unknown_cols = ['Education_Level', 'Income_Category', 'Marital_Status']

results = []
for col in unknown_cols:
    is_unknown = df[col] == 'Unknown'
    contingency = pd.crosstab(is_unknown, df['Churn'])
    chi2, p_value, _, _ = chi2_contingency(contingency)
    results.append({
        'column': col,
        'n_unknown': int(is_unknown.sum()),
        'pct_unknown': round(100 * is_unknown.mean(), 1),
        'churn_if_unknown': round(df.loc[is_unknown, 'Churn'].mean(), 4),
        'churn_if_known': round(df.loc[~is_unknown, 'Churn'].mean(), 4),
        'chi2_p_value': round(p_value, 4),
    })

pd.DataFrame(results)

,column,n_unknown,pct_unknown,churn_if_unknown,churn_if_known,chi2_p_value
0,Education_Level,1519,15.0,0.1685,0.1593,0.3852
1,Income_Category,1112,11.0,0.1682,0.1597,0.4970
2,Marital_Status,749,7.4,0.1722,0.1597,0.3985


**The missingness is unrelated to churn.**

For all three columns the churn rate among customers with an `Unknown` value is
within about one percentage point of the rate among those without, and every
chi-square p-value is far above 0.05 (0.39, 0.50, 0.40). There is no evidence
that withholding a demographic detail says anything about whether a customer
leaves.

So option 3 is safe, and it is taken: `Unknown` becomes a genuine missing value.
No imputation is performed anywhere in this project.

This matters for honesty of the output. When notebook 03 reports a churn rate by
education level, that rate is computed from the 8,608 customers whose education
is actually recorded, and the notebook says so. The alternative - filling 1,519
customers with "Graduate" because it is the mode - would produce a chart that
looks more complete and is less true.

In [9]:
for col in unknown_cols:
    df[col] = df[col].replace('Unknown', np.nan)

missing = df.isna().sum()
missing[missing > 0].to_frame('missing values now visible')

,missing values now visible
Education_Level,1519
Marital_Status,749
Income_Category,1112


The missing values are now visible to pandas, which means they will be excluded
from grouped statistics automatically instead of silently forming their own
category.

---

# Section 5: Give the ordered categories an order

`Education_Level`, `Income_Category` and `Card_Category` are ordinal - their
values have a natural sequence. Left as plain text, pandas sorts them
alphabetically, which puts `$120K +` before `Less than $40K` and makes any chart
of income against churn nonsense to read.

In [10]:
education_order = ['Uneducated', 'High School', 'College',
                   'Graduate', 'Post-Graduate', 'Doctorate']

income_order = ['Less than $40K', '$40K - $60K', '$60K - $80K',
                '$80K - $120K', '$120K +']

card_order = ['Blue', 'Silver', 'Gold', 'Platinum']

df['Education_Level'] = pd.Categorical(df['Education_Level'],
                                       categories=education_order, ordered=True)
df['Income_Category'] = pd.Categorical(df['Income_Category'],
                                       categories=income_order, ordered=True)
df['Card_Category'] = pd.Categorical(df['Card_Category'],
                                     categories=card_order, ordered=True)

for col in ['Education_Level', 'Income_Category', 'Card_Category']:
    print(f"{col}: {list(df[col].cat.categories)}")

Education_Level: ['Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate']
Income_Category: ['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +']
Card_Category: ['Blue', 'Silver', 'Gold', 'Platinum']


In [11]:
df.groupby('Income_Category', observed=False)['Churn'].agg(['mean', 'size']).round(4)

,mean,size
Income_Category,,
Less than $40K,0.1719,3561
$40K - $60K,0.1514,1790
$60K - $80K,0.1348,1402
$80K - $120K,0.1577,1535
$120K +,0.1733,727


Income now runs in the correct order, and the `Unknown` customers are excluded
rather than sitting between two income bands.

---

# Section 6: Two problems that cannot be cleaned

Not every data quality issue has a fix. Two are recorded here so that the
findings in notebook 03 can be read with them in mind.

In [12]:
on_book = df['Months_on_book'].value_counts().sort_values(ascending=False)
print("Most common Months_on_book values:")
print(on_book.head(6).to_string())
print()
print(f"Customers at exactly 36 months: {(df['Months_on_book'] == 36).sum():,} "
      f"({(df['Months_on_book'] == 36).mean():.1%})")

Most common Months_on_book values:
Months_on_book
36    2463
37     358
34     353
38     347
39     341
40     333

Customers at exactly 36 months: 2,463 (24.3%)


**`Months_on_book` has a spike at exactly 36.** Almost a quarter of all
customers share that single value, while neighbouring values hold around 350
each. A genuine tenure distribution does not do this. It looks like a default
that was written in wherever the real tenure was not available.

There is no way to recover the true values, and no way to tell which of the
2,463 are real. `Months_on_book` is therefore treated as unreliable and is not
used to draw any conclusion in this project.

In [13]:
print("Correlation, Customer_Age vs Months_on_book: "
      f"{df['Customer_Age'].corr(df['Months_on_book']):.4f}")

Correlation, Customer_Age vs Months_on_book: 0.7889


**`Months_on_book` is also largely redundant.** It correlates 0.79 with
`Customer_Age`, which is close enough that the two say much the same thing.
Reporting both as separate findings would be double counting.

Neither column is dropped - removing data because it is inconvenient is worse
than keeping it and stating the caveat - but both facts are carried into the
findings.

---

# Section 7: Validate and save

In [14]:
print(f"Rows:    {df.shape[0]:,}  (unchanged from {raw.shape[0]:,})")
print(f"Columns: {df.shape[1]}   (from {raw.shape[1]})")
print()
print("No rows were dropped at any point in this notebook.")
print(f"Duplicated rows: {df.duplicated().sum()}")
print(f"Churn rate:      {df['Churn'].mean():.4f}")

Rows:    10,127  (unchanged from 10,127)
Columns: 20   (from 23)

No rows were dropped at any point in this notebook.
Duplicated rows: 0
Churn rate:      0.1607


In [15]:
df.dtypes.to_frame('dtype')

,dtype
Attrition_Flag,object
Customer_Age,int64
Gender,object
Dependent_count,int64
Education_Level,category
Marital_Status,object
Income_Category,category
Card_Category,category
Months_on_book,int64
Total_Relationship_Count,int64


In [16]:
os.makedirs('outputs/datasets/cleaned', exist_ok=True)
output_path = 'outputs/datasets/cleaned/bank_churners_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

reloaded = pd.read_csv(output_path)
print(f"Reloaded shape: {reloaded.shape}")
print(f"Reloaded churn rate: {reloaded['Churn'].mean():.4f}")

Saved to outputs/datasets/cleaned/bank_churners_cleaned.csv
Reloaded shape: (10127, 20)
Reloaded churn rate: 0.1607


The reloaded file matches, so the saved dataset is what notebook 03 will work
from.

One caveat worth knowing: CSV does not preserve the ordered categories set in
section 5. Notebook 03 reapplies them on load rather than assuming they
survived.

---

# Conclusions and Next Steps

**What was changed:**

* Dropped `CLIENTNUM` (an identifier), `Avg_Open_To_Buy` (exactly derivable from
  two other columns) and the two `Naive_Bayes_Classifier` columns (target
  leakage).
* Added `Churn` as a 0/1 version of `Attrition_Flag`.
* Converted `Unknown` to a real missing value in the three demographic columns.
* Applied a meaningful order to the three ordinal categorical columns.

**What was deliberately not changed:**

* No rows were dropped. All 10,127 customers survive.
* No values were imputed. The chi-square tests showed the missingness is
  unrelated to churn, so excluding those customers from demographic comparisons
  is safe, and filling them in would have invented data.
* No outliers were removed. The extreme `Credit_Limit` values are real customers
  with high limits, not errors.

**Caveats carried forward into the analysis:**

* `Months_on_book` has a default-looking spike at 36 covering 24.3% of
  customers, and correlates 0.79 with `Customer_Age`. Not used for conclusions.
* Demographic charts are computed on the customers whose attribute is recorded,
  so their denominators are smaller than 10,127. Each chart states its own.

Next: notebook 03, exploratory data analysis.